<a href="https://colab.research.google.com/github/kavanabt-07/ai-based-fake-news-detection/blob/main/ai_based_fake_news_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==========================================
# 1. INSTALL AND IMPORT REQUIRED LIBRARIES
# ==========================================
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Download NLTK stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

print("Libraries imported successfully!\n")

# ==========================================
# 2. CREATE A SAMPLE DATASET
# (Replace this section with pd.read_csv() if using your own data)
# ==========================================
data = {
    'title': [
        'Breaking: Scientists discover cure for aging in red wine',
        'Government announces new tax reduction policy for small businesses',
        'Aliens spotted landing near the Eiffel Tower in Paris',
        'NASA launches new space telescope to study distant galaxies',
        'Drinking boiling water cures all viral infections instantly',
        'Central Bank updates interest rates following economic review',
        'Famous celebrity secretly running underground criminal ring',
        'Global health organization releases updated nutrition guidelines'
    ],
    'label': ['FAKE', 'REAL', 'FAKE', 'REAL', 'FAKE', 'REAL', 'FAKE', 'REAL']
}

df = pd.DataFrame(data)
print("--- Sample Dataset ---")
print(df.head(), "\n")

# ==========================================
# 3. TEXT PREPROCESSING FUNCTION
# ==========================================
def preprocess_text(text):
    # Remove non-alphabetic characters
    text = re.sub('[^a-zA-Z]', ' ', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize and remove stop words & apply stemming
    words = text.split()
    words = [ps.stem(word) for word in words if word not in stop_words]
    return ' '.join(words)

# Apply preprocessing
df['cleaned_title'] = df['title'].apply(preprocess_text)

# ==========================================
# 4. TRAIN / TEST SPLIT & TF-IDF VECTORIZATION
# ==========================================
X = df['cleaned_title']
y = df['label']

# Split dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert text into numerical TF-IDF vectors
vectorizer = TfidfVectorizer()
tfidf_train = vectorizer.fit_transform(X_train)
tfidf_test = vectorizer.transform(X_test)

# ==========================================
# 5. MODEL TRAINING & EVALUATION
# ==========================================
model = PassiveAggressiveClassifier(max_iter=50)
model.fit(tfidf_train, y_train)

# Make predictions on test set
y_pred = model.predict(tfidf_test)

# Print performance metrics
accuracy = accuracy_score(y_test, y_pred)
print("--- Model Evaluation ---")
print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ==========================================
# 6. INTERACTIVE PREDICTION SYSTEM
# ==========================================
def predict_news(news_text):
    cleaned = preprocess_text(news_text)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    return prediction

print("\n--- Testing Custom News Inputs ---")
sample_news_1 = "Secret study reveals drinking lemon tea eliminates all viruses in 2 hours."
sample_news_2 = "The federal reserve held its quarterly meeting to discuss inflation target rates."

print(f"News: '{sample_news_1}'")
print(f"Prediction: {predict_news(sample_news_1)}\n")

print(f"News: '{sample_news_2}'")
print(f"Prediction: {predict_news(sample_news_2)}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Libraries imported successfully!

--- Sample Dataset ---
                                               title label
0  Breaking: Scientists discover cure for aging i...  FAKE
1  Government announces new tax reduction policy ...  REAL
2  Aliens spotted landing near the Eiffel Tower i...  FAKE
3  NASA launches new space telescope to study dis...  REAL
4  Drinking boiling water cures all viral infecti...  FAKE 

--- Model Evaluation ---
Accuracy: 100.00%

Classification Report:
               precision    recall  f1-score   support

        REAL       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2


--- Testing Custom News Inputs ---
News: 'Secret study reveals drinking lemon tea eliminates all viruses in 2 hours.'
Prediction: FAKE

News: 'The federal reserve held its quarterly meeting to discuss inflation target rates.'
Prediction: FAKE
